In [1]:
%load_ext autoreload
%autoreload 2

In [11]:
import time

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.operators.crossover.ox import OrderCrossover
from pymoo.operators.mutation.inversion import InversionMutation
from pymoo.operators.sampling.rnd import PermutationRandomSampling
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.termination.collection import TerminationCollection
from pymoo.termination.default import DefaultSingleObjectiveTermination
from tqdm import tqdm

from qap_assignment.dataset import make_dataset, parse_dat_file
from qap_assignment.operators import SwapMutation, TabuSearchRepair
from qap_assignment.problem import QAP

make_dataset()

In [36]:
def run_kra(seed):
    n, A, B = parse_dat_file("kra30a")
    problem = QAP(n, A, B)

    algorithm = GA(
        pop_size=50,
        sampling=PermutationRandomSampling(),
        # `Crossover` uses prob=0.9 by default anyway
        crossover=OrderCrossover(prob=0.9),
        mutation=SwapMutation(prob=0.2),
        # By default, tournament selection and elitist survival is used
        repair=TabuSearchRepair(),
    )

    termination = TerminationCollection(
        DefaultSingleObjectiveTermination(
            period=30,  # Don't think this matters much since we have local search
            n_max_gen=50000,
            # Make n_max_evals really big since I don't care about this
            n_max_evals=2**64,
        ),
        # Why is fmin not documented...
        # Shouldn't hardcode this, kra30a has a known optimum
        get_termination("fmin", 88900),
    )

    start_time = time.perf_counter()
    res = minimize(problem, algorithm, termination, seed=seed, verbose=True)
    end_time = time.perf_counter()
    return {
        "cost": res.F[0],
        "time": end_time - start_time,
    }

In [37]:
seed_sequence = np.random.SeedSequence(42)
run_seeds = seed_sequence.generate_state(50)
tasks = (delayed(run_kra)(seed) for seed in run_seeds)
kra_results = []
for res in tqdm(
    Parallel(n_jobs=8, return_as="generator_unordered")(tasks),
    total=50,
):
    kra_results.append(res)

100%|██████████| 50/50 [00:14<00:00,  3.36it/s]


In [38]:
avg_cost = np.average([r["cost"] for r in kra_results])
avg_time = np.average([r["time"] for r in kra_results])
print(avg_cost)
print(avg_time)
print(np.min([r["cost"] for r in kra_results]))

88900.0
0.5305936199845747
88900.0


In [ ]:
def run_tai(seed):
    n, A, B = parse_dat_file("tai40a")
    problem = QAP(n, A, B)

    algorithm = GA(
        pop_size=100,
        sampling=PermutationRandomSampling(),
        # `Crossover` uses prob=0.9 by default anyway
        crossover=OrderCrossover(prob=0.95),
        mutation=SwapMutation(
            prob=0.4
        ),  # 0.4 swap prob, 0.95 crossover is the best i've gotten so far
        # By default, tournament selection and elitist survival is used
        repair=TabuSearchRepair(),
    )

    termination = TerminationCollection(
        DefaultSingleObjectiveTermination(
            period=30,  # Don't think this matters much since we have local search
            n_max_gen=50000,
            # Make n_max_evals really big since I don't care about this
            n_max_evals=2**64,
        ),
    )

    start_time = time.perf_counter()
    res = minimize(problem, algorithm, termination, seed=seed, verbose=True)
    end_time = time.perf_counter()
    return {
        "cost": res.F[0],
        "time": end_time - start_time,
    }

In [39]:
seed_sequence = np.random.SeedSequence(42)
run_seeds = seed_sequence.generate_state(50)
tasks = (delayed(run_tai)(seed) for seed in run_seeds)
tai_results = []
for res in tqdm(
    Parallel(n_jobs=8, return_as="generator_unordered")(tasks),
    total=50,
):
    tai_results.append(res)

100%|██████████| 50/50 [01:20<00:00,  1.61s/it]


In [40]:
avg_cost = np.average([r["cost"] for r in tai_results])
avg_time = np.average([r["time"] for r in tai_results])
print(avg_cost)
print(avg_time)
print(np.min([r["cost"] for r in tai_results]))

3153351.0
12.165538860009983
3141702.0
